# Meth3D-Net V6 — TCGA PanCancer Validation (Colab)
## All chromosomes | 105,451 probes × 9,854 samples | From Google Drive

**Account:** neetushrinate0406@gmail.com

### Files read from `MyDrive/Meth3DNet_TCGA/`:
| File | Size | Role |
|---|---|---|
| `tcga_dmb_probes_full.tsv` | 12.8 GB | Pre-extracted — FAST path |
| `tcga_dmb_probes_q_only.tsv` | 8.1 GB | Q-arm alternative |
| `jhu-usc.edu_PANCAN...betaValue.tsv` | 49.2 GB | Raw fallback |
| `humanmethylation450_15017482_v1-2.csv` | 197 MB | Illumina manifest |
| `Methylation_Paper_CpG_v6/` | — | V6 DMB + CT score CSVs |

### Run order:
1. Runtime → **Change runtime type → High RAM** (25 GB)
2. Run Cell 0 first (mounts Drive)
3. Run All remaining cells

### Three validation layers:
| Layer | Test | Key metric |
|---|---|---|
| A | High V6 \|Δβ\| probes more variable in TCGA? | Fold + Mann-Whitney p |
| B | V6 CT-z instability predicts TCGA variance? | Spearman r |
| C | V6 Δβ concordant with tumour vs normal? | Concordance % |


## Cell 0 — Mount Google Drive

In [1]:
# Mount Google Drive — run this first
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')
print('Account: neetushrinate0406@gmail.com')


Mounted at /content/drive
Drive mounted.
Account: neetushrinate0406@gmail.com


## Cell 0 — Setup and file discovery

In [2]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import spearmanr, mannwhitneyu
warnings.filterwarnings('ignore')

# ── Drive paths ───────────────────────────────────────────
TCGA_BASE_PATH = '/content/drive/MyDrive/Meth3DNet_TCGA'
V6_BASE_PATH   = f'{TCGA_BASE_PATH}/Methylation_Paper_CpG_v6'
OUT_DIR        = f'{TCGA_BASE_PATH}/tcga_validation_results'
WORK_DIR       = '/content'
os.makedirs(OUT_DIR, exist_ok=True)

CHROMS = [f'chr{i}' for i in list(range(1,23))+['X','Y']]

# ── V6 DMB file finder ────────────────────────────────────
V6_ROOTS = []
for root, dirs, files in os.walk(V6_BASE_PATH):
    if any('_V6_dmb_' in f for f in files):
        V6_ROOTS.append(root)
V6_ROOTS = sorted(set(V6_ROOTS))

def _find_v6_file(chrom, suffix):
    for d in V6_ROOTS:
        p = os.path.join(d, f'{chrom}_V6_{suffix}')
        if os.path.exists(p): return p
    return None

# ── General file finder (handles spaces, (1) in names) ────
def find_file(keywords, root=TCGA_BASE_PATH):
    best = None
    for dp, _, files in os.walk(root):
        for fn in files:
            fn_low = fn.lower()
            if all(k.lower() in fn_low for k in keywords):
                fp = os.path.join(dp, fn)
                if best is None or '(1)' not in fn:
                    best = fp
    return best

# ── Discover TCGA files ───────────────────────────────────
TCGA_PRE  = find_file(['tcga_dmb_probes_full'])
TCGA_QARM = find_file(['tcga_dmb_probes_q_only'])
TCGA_RAW  = find_file(['humanmethylation450', 'betavalue'])
MANIFEST  = find_file(['humanmethylation450', '15017482'])

n_v6 = sum(1 for c in CHROMS if _find_v6_file(c,'dmb_p.csv') is not None)

print('=== Meth3D-Net V6 TCGA Validation (Colab) ===')
print(f'Account            : neetushrinate0406@gmail.com')
print(f'V6 DMB directories : {len(V6_ROOTS)}')
for d in V6_ROOTS: print(f'  {d}')
print(f'V6 DMB files found : {n_v6}/24')
print()
for label, path in [
    ('Pre-extracted (full)', TCGA_PRE),
    ('Pre-extracted (q-arm)', TCGA_QARM),
    ('Raw TCGA (49 GB)',      TCGA_RAW),
    ('Manifest (197 MB)',     MANIFEST),
]:
    if path and os.path.exists(path):
        sz = os.path.getsize(path)
        unit = 'GB' if sz > 1e9 else 'MB'
        szv = sz/1e9 if sz > 1e9 else sz/1e6
        print(f'  FOUND   {label:<28}: {szv:.1f} {unit}')
        print(f'          {os.path.basename(path)}')
    else:
        print(f'  MISSING {label}')
print()

if TCGA_PRE:
    TCGA_USE = TCGA_PRE
    print('Mode: FAST — tcga_dmb_probes_full.tsv (~10 min load)')
elif TCGA_QARM:
    TCGA_USE = TCGA_QARM
    print('Mode: Q-ARM — tcga_dmb_probes_q_only.tsv')
elif TCGA_RAW:
    TCGA_USE = None
    print('Mode: SCAN — scanning raw 49 GB file (20-60 min)')
else:
    TCGA_USE = None
    print('WARNING: No TCGA file found! Check TCGA_BASE_PATH.')


=== Meth3D-Net V6 TCGA Validation (Colab) ===
Account            : neetushrinate0406@gmail.com
V6 DMB directories : 2
  /content/drive/MyDrive/Meth3DNet_TCGA/Methylation_Paper_CpG_v6
  /content/drive/MyDrive/Meth3DNet_TCGA/Methylation_Paper_CpG_v6/Methylation Paper cpG_v6
V6 DMB files found : 24/24

  FOUND   Pre-extracted (full)        : 12.8 GB
          tcga_dmb_probes_full.tsv
  FOUND   Pre-extracted (q-arm)       : 8.1 GB
          tcga_dmb_probes_q_only.tsv
  FOUND   Raw TCGA (49 GB)            : 49.2 GB
          jhu-usc.edu_PANCAN_HumanMethylation450.betaValue.tsv
  FOUND   Manifest (197 MB)           : 197.1 MB
          humanmethylation450_15017482_v1-2.csv

Mode: FAST — tcga_dmb_probes_full.tsv (~10 min load)


## Cell 1 — Load V6 DMBs (all chromosomes)

In [3]:
def safe_read_csv(fp):
    try:
        df = pd.read_csv(fp)
        return df if len(df) > 0 else None
    except (pd.errors.EmptyDataError, pd.errors.ParserError):
        return None
    except Exception as e:
        print(f'  Skip {os.path.basename(fp)}: {e}')
        return None

print('Loading V6 DMBs...')
all_dmbs, empty_arms, loaded = [], [], []
for chrom in CHROMS:
    for arm in ['p', 'q']:
        fp = _find_v6_file(chrom, f'dmb_{arm}.csv')
        if not fp:
            empty_arms.append(f'{chrom}_{arm}')
            continue
        df = safe_read_csv(fp)
        if df is not None:
            df['chrom'] = chrom
            df['arm']   = arm
            all_dmbs.append(df)
            loaded.append(f'{chrom}_{arm}')
        else:
            empty_arms.append(f'{chrom}_{arm}')

if not all_dmbs:
    raise RuntimeError('No V6 DMB files found. '
                       'Add methylation-paper-cpg-v6 dataset.')

dmb_df = pd.concat(all_dmbs, ignore_index=True)
if 'abs_delta' not in dmb_df.columns:
    dmb_df['abs_delta'] = dmb_df['delta'].abs()
if 'ram_sig_99' not in dmb_df.columns:
    dmb_df['ram_sig_99'] = dmb_df['abs_delta'] > 0.20
if 'start' not in dmb_df.columns:
    dmb_df['start'] = (dmb_df['mid_mb'] * 1e6).astype(int)
    dmb_df['end']   = dmb_df['start'] + 50_000

sig_dmb = dmb_df[dmb_df['ram_sig_99']].copy().reset_index(drop=True)

print(f'Arms loaded          : {len(loaded)}')
print(f'Arms empty/missing   : {empty_arms}')
print(f'V6 DMBs (total)      : {len(dmb_df):,}')
print(f'Significant (RaM-99) : {len(sig_dmb):,}')
print(f'Mean |Δβ|            : {sig_dmb["abs_delta"].mean():.4f}')

# Build sorted interval lookup per chromosome
dmb_intervals = {}
for chrom in CHROMS:
    sub = sig_dmb[sig_dmb['chrom']==chrom][
        ['start','end','delta','abs_delta']].copy()
    if len(sub):
        sub = sub.sort_values('start').reset_index(drop=True)
        dmb_intervals[chrom] = sub
print(f'Chromosomes with DMBs: {sorted(dmb_intervals.keys())}')


Loading V6 DMBs...
Arms loaded          : 42
Arms empty/missing   : ['chr13_p', 'chr14_p', 'chr15_p', 'chr22_p', 'chrY_p', 'chrY_q']
V6 DMBs (total)      : 213,347
Significant (RaM-99) : 77,066
Mean |Δβ|            : 0.3182
Chromosomes with DMBs: ['chr1', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr2', 'chr20', 'chr21', 'chr22', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chrX']


## Cell 2 — Load Illumina 450K manifest and intersect probes with V6 DMBs
Manifest has 485,512 valid probes. Intersection gives DMB-overlapping probe set.

In [4]:
print('Loading 450K manifest...')

if not MANIFEST or not os.path.exists(MANIFEST):
    raise FileNotFoundError(
        'humanmethylation450_15017482_v1-2.csv not found.\n'
        'It should be in the tcga-pancan-methylation dataset.'
    )

# Detect header row (skips Illumina metadata lines)
header_row = 0
with open(MANIFEST, 'r', encoding='utf-8', errors='ignore') as f:
    for i, line in enumerate(f):
        if line.startswith('IlmnID') or line.startswith('Name'):
            header_row = i
            break
print(f'  Header at row: {header_row}')

manifest_df = pd.read_csv(
    MANIFEST, skiprows=header_row, low_memory=False,
    usecols=lambda c: c in [
        'Name','IlmnID','CHR','MAPINFO',
        'Chromosome_36','Position_36','Chr','MapInfo'
    ]
)
name_col = next((c for c in manifest_df.columns if c in ['Name','IlmnID']), None)
chr_col  = next((c for c in manifest_df.columns
                 if c in ['CHR','Chr','Chromosome_36']), None)
pos_col  = next((c for c in manifest_df.columns
                 if c in ['MAPINFO','MapInfo','Position_36']), None)

print(f'  Columns: probe={name_col}  chr={chr_col}  pos={pos_col}')
print(f'  Manifest shape: {manifest_df.shape}')

manifest_df = manifest_df[[name_col, chr_col, pos_col]].copy().dropna()
manifest_df.columns = ['probe_id','chrom','position']
manifest_df['chrom'] = (
    'chr' + manifest_df['chrom'].astype(str)
    .str.replace('chr','',regex=False).str.strip()
)
manifest_df['position'] = pd.to_numeric(
    manifest_df['position'], errors='coerce')
manifest_df = manifest_df.dropna(subset=['position'])
manifest_df['position'] = manifest_df['position'].astype(int)
print(f'  Valid probes: {len(manifest_df):,}')

# ── Intersect with V6 DMB windows ────────────────────────────
print('\nIntersecting 450K probes with V6 DMB windows...')
probe_dmb_map = {}
for chrom, intervals in dmb_intervals.items():
    chrom_probes = manifest_df[manifest_df['chrom']==chrom]
    if len(chrom_probes) == 0: continue
    starts = intervals['start'].values
    ends   = intervals['end'].values
    deltas = intervals['delta'].values
    abs_d  = intervals['abs_delta'].values
    for _, row in chrom_probes.iterrows():
        pos = row['position']
        idx = np.searchsorted(starts, pos, side='right') - 1
        if idx >= 0 and starts[idx] <= pos <= ends[idx]:
            probe_dmb_map[row['probe_id']] = {
                'chrom'        : chrom,
                'position'     : int(pos),
                'v6_delta'     : float(deltas[idx]),
                'v6_abs_delta' : float(abs_d[idx]),
            }

DMB_PROBE_SET = set(probe_dmb_map.keys())
print(f'Probes overlapping V6 DMBs : {len(DMB_PROBE_SET):,}')
print(f'  ({len(DMB_PROBE_SET)/len(manifest_df)*100:.1f}% of 450K probes)')

# Delta distribution
deltas_arr = np.array([v['v6_abs_delta'] for v in probe_dmb_map.values()])
print(f'|Δβ| distribution  : mean={deltas_arr.mean():.3f} '
      f'median={np.median(deltas_arr):.3f} '
      f'max={deltas_arr.max():.3f}')


Loading 450K manifest...
  Header at row: 7
  Columns: probe=IlmnID  chr=CHR  pos=MAPINFO
  Manifest shape: (486428, 5)
  Valid probes: 485,512

Intersecting 450K probes with V6 DMB windows...
Probes overlapping V6 DMBs : 134,178
  (27.6% of 450K probes)
|Δβ| distribution  : mean=0.258 median=0.240 max=0.662


## Cell 3 — Load TCGA methylation matrix
Fast path: loads `tcga_dmb_probes_full.tsv` in float32 chunks (~3.2 GB RAM).
Slow path: scans raw 49 GB file line-by-line (20–60 min, low RAM).

In [5]:
DEMO_TCGA = False

if TCGA_USE and os.path.exists(TCGA_USE):
    sz_gb = os.path.getsize(TCGA_USE)/1e9
    fname = os.path.basename(TCGA_USE)
    print(f'Loading: {fname}  ({sz_gb:.1f} GB)')
    print('Reading in float32 chunks to manage RAM...')

    CHUNK = 5000
    chunks, n = [], 0
    for chunk in pd.read_csv(TCGA_USE, sep='\t',
                              index_col=0, low_memory=False,
                              chunksize=CHUNK):
        n += 1
        if n % 4 == 0:
            print(f'  {n*CHUNK:,} probes...', end='\r')
        chunks.append(chunk.astype('float32'))

    tcga_meth = pd.concat(chunks)
    del chunks
    tcga_meth.columns = [c[:15] for c in tcga_meth.columns]
    mem = tcga_meth.memory_usage(deep=True).sum()/1e9
    print(f'\nShape  : {tcga_meth.shape}')
    print(f'Memory : {mem:.2f} GB')
    print(f'Samples: {list(tcga_meth.columns[:3])} ...')
    DEMO_TCGA = False

elif TCGA_RAW and os.path.exists(TCGA_RAW):
    sz_gb = os.path.getsize(TCGA_RAW)/1e9
    print(f'Scanning raw file ({sz_gb:.1f} GB) in low-RAM mode...')
    print('Writing matching probes directly to disk. ~20-60 min.')
    tmp = '/content/tcga_scan_tmp.tsv'
    found, total = 0, 0
    with open(TCGA_RAW,'rt',errors='replace') as fh, \
         open(tmp,'wt') as out:
        hdr = fh.readline().rstrip('\n')
        sep = '\t' if '\t' in hdr else ','
        sids = [s.strip()[:15] for s in hdr.split(sep)[1:]]
        out.write('probe_id\t' + '\t'.join(sids) + '\n')
        for line in fh:
            total += 1
            if total % 100_000 == 0:
                print(f'  {total:,} lines | {found:,} probes',
                      end='\r', flush=True)
            parts = line.rstrip('\n').split(sep)
            pid = parts[0].strip().strip('"')
            if pid in DMB_PROBE_SET:
                n = len(sids)
                v = parts[1:]
                out.write(pid+'\t'+'\t'.join(
                    v[:n]+['NA']*(n-len(v[:n])))+'\n')
                found += 1
            if found >= len(DMB_PROBE_SET): break
    print(f'\nScanned {total:,} | found {found:,}')
    tcga_meth = pd.read_csv(tmp,sep='\t',index_col=0).astype('float32')
    tcga_meth.columns = [c[:15] for c in tcga_meth.columns]
    print(f'Shape: {tcga_meth.shape}')
    DEMO_TCGA = False
else:
    raise RuntimeError(
        'No TCGA file found. Check that one of these exists in '
        f'{TCGA_BASE_PATH}:\n'
        '  tcga_dmb_probes_full.tsv\n'
        '  tcga_dmb_probes_q_only.tsv\n'
        '  jhu-usc.edu_PANCAN_HumanMethylation450.betaValue.tsv'
    )

# Overlap report
probes_in_tcga = DMB_PROBE_SET & set(tcga_meth.index)
print(f'\nDMB probes in TCGA : {len(probes_in_tcga):,} / {len(DMB_PROBE_SET):,}'
      f' ({len(probes_in_tcga)/max(len(DMB_PROBE_SET),1)*100:.1f}% retention)')


Loading: tcga_dmb_probes_full.tsv  (12.8 GB)
Reading in float32 chunks to manage RAM...

Shape  : (105451, 9854)
Memory : 4.16 GB
Samples: ['TCGA-OR-A5J1-01', 'TCGA-OR-A5J2-01', 'TCGA-OR-A5J3-01'] ...

DMB probes in TCGA : 105,451 / 134,178 (78.6% retention)


## Cell 4 — Layer A: Variance Enrichment

In [6]:
print('=== Layer A: V6 DMB Enrichment in TCGA Variable CpGs ===')
print()
print('Strategy: split 105,451 extracted probes by V6 |Δβ| strength')
print('  High |Δβ| (≥0.30) = core DMB probes (strong instability)')
print('  Low  |Δβ| (<0.30) = DMB-adjacent probes (weaker signal)')
print('  This mirrors the MB validation (fold=1.62×, p<1e-300)')
print()

# Variance per probe across all 9,854 TCGA samples
probe_var = tcga_meth.var(axis=1).dropna()
print(f'TCGA variance computed: {len(probe_var):,} probes')
print(f'  Range  : [{probe_var.min():.5f}, {probe_var.max():.5f}]')
print(f'  Median : {probe_var.median():.5f}')
print()

# Map |delta| to each probe
abs_delta_map = pd.Series({
    p: probe_dmb_map[p]['v6_abs_delta']
    for p in probe_var.index
    if p in probe_dmb_map
})
common = probe_var.index.intersection(abs_delta_map.index)
pv_common = probe_var.reindex(common)
ad_common = abs_delta_map.reindex(common)

# Split into high vs low |delta|
high_mask = ad_common >= 0.30
low_mask  = ad_common <  0.30
var_high  = pv_common[high_mask]
var_low   = pv_common[low_mask]

print(f'High |Δβ| probes (≥0.30): n={len(var_high):,}')
print(f'  Median variance: {var_high.median():.5f}')
print(f'  Mean   variance: {var_high.mean():.5f}')
print(f'Low  |Δβ| probes (<0.30): n={len(var_low):,}')
print(f'  Median variance: {var_low.median():.5f}')
print(f'  Mean   variance: {var_low.mean():.5f}')
print()

# Mann-Whitney U test: high > low
if len(var_high) > 10 and len(var_low) > 10:
    stat_a, pval = mannwhitneyu(var_high, var_low, alternative='greater')
    median_high = var_high.median()
    median_low  = var_low.median()
    fold_enrich = median_high / max(median_low, 1e-9)

    print(f'Fold enrichment (median) : {fold_enrich:.3f}x')
    print(f'Mann-Whitney p-value     : {pval:.3e}')

    # Also compute Spearman r(|delta|, variance)
    from scipy.stats import spearmanr as _sp
    r_av, p_av = _sp(ad_common, pv_common)
    print(f'Spearman r(|Δβ|, var)    : {r_av:.4f}  p={p_av:.3e}')
    print()

    if pval < 0.001 and fold_enrich > 1.3:
        print('RESULT: ✔ STRONG — High V6 |Δβ| probes are significantly'
              ' more variable in TCGA')
    elif pval < 0.05:
        print('RESULT: ~ Significant enrichment')
    else:
        print('RESULT: ✗ Not significant')
else:
    print('Insufficient probes for comparison.')
    fold_enrich, pval = np.nan, np.nan

# Save
layerA_summary = pd.DataFrame([{
    'n_high_delta'    : int(len(var_high)),
    'n_low_delta'     : int(len(var_low)),
    'median_var_high' : round(float(var_high.median()), 6),
    'median_var_low'  : round(float(var_low.median()), 6),
    'fold_enrichment' : round(float(fold_enrich), 4),
    'mannwhitney_p'   : float(pval),
    'spearman_r_delta_var': round(float(r_av), 4),
    'spearman_p'      : float(p_av),
}])
layerA_summary.to_csv(f'{OUT_DIR}/layerA_summary.csv', index=False)
print(layerA_summary.T.to_string())

layerA_detail = pd.DataFrame({
    'probe_id'     : common,
    'v6_abs_delta' : ad_common.values,
    'delta_group'  : ['High(≥0.30)' if h else 'Low(<0.30)'
                      for h in high_mask.values],
    'tcga_variance': pv_common.values,
})
layerA_detail.to_csv(f'{OUT_DIR}/layerA_enrichment.csv', index=False)
print('Saved: layerA_summary.csv, layerA_enrichment.csv')


=== Layer A: V6 DMB Enrichment in TCGA Variable CpGs ===

Strategy: split 105,451 extracted probes by V6 |Δβ| strength
  High |Δβ| (≥0.30) = core DMB probes (strong instability)
  Low  |Δβ| (<0.30) = DMB-adjacent probes (weaker signal)
  This mirrors the MB validation (fold=1.62×, p<1e-300)

TCGA variance computed: 105,451 probes
  Range  : [0.00000, 0.12599]
  Median : 0.03035

High |Δβ| probes (≥0.30): n=27,349
  Median variance: 0.03752
  Mean   variance: 0.03773
Low  |Δβ| probes (<0.30): n=78,102
  Median variance: 0.02762
  Mean   variance: 0.02900

Fold enrichment (median) : 1.359x
Mann-Whitney p-value     : 0.000e+00
Spearman r(|Δβ|, var)    : 0.2712  p=0.000e+00

RESULT: ✔ STRONG — High V6 |Δβ| probes are significantly more variable in TCGA
                                 0
n_high_delta          27349.000000
n_low_delta           78102.000000
median_var_high           0.037524
median_var_low            0.027620
fold_enrichment           1.358600
mannwhitney_p             0.000

## Cell 5 — Layer B: CT Instability vs TCGA Variance

In [7]:
print('=== Layer B: V6 CT Instability Score vs TCGA Variance ===')

ct_rows = []
for chrom in CHROMS:
    fp = _find_v6_file(chrom, 'ct_scores.csv')
    if not fp: continue
    df = safe_read_csv(fp)
    if df is not None:
        df['chrom'] = chrom
        ct_rows.append(df)

r_b, p_b, layerB = np.nan, np.nan, pd.DataFrame()

if not ct_rows:
    print('No CT score files found — Layer B skipped.')
else:
    ct_df = pd.concat(ct_rows, ignore_index=True)
    print(f'CT windows loaded: {len(ct_df):,} across '
          f'{ct_df.chrom.nunique()} chromosomes')

    # Identify CT value column
    ct_col = next((c for c in ct_df.columns
                   if 'ct' in c.lower() and 'score' in c.lower()), None)
    if ct_col is None:
        skip = {'chrom','arm','start','end','mid_mb'}
        ct_col = next((c for c in ct_df.columns if c not in skip and
                       pd.to_numeric(ct_df[c],errors='coerce')
                       .notna().sum() > 100), None)

    if 'start' not in ct_df.columns and 'mid_mb' in ct_df.columns:
        ct_df['start'] = (ct_df['mid_mb']*1e6).astype(int)
        ct_df['end']   = ct_df['start'] + 50_000

    if ct_col is None:
        print(f'CT column not found. Columns: {list(ct_df.columns)}')
    else:
        # Z-score CT values
        ct_mean = ct_df[ct_col].mean()
        ct_std  = ct_df[ct_col].std()
        ct_df['ct_z'] = (ct_df[ct_col] - ct_mean) / max(ct_std, 1e-9)
        print(f'CT column   : {ct_col}')
        print(f'CT-z range  : [{ct_df.ct_z.min():.2f}, {ct_df.ct_z.max():.2f}]')

        # ── Vectorised probe → CT window mapping ─────────────
        # Build probe position table from probe_dmb_map
        probe_pos_df = pd.DataFrame([
            {'probe_id': p,
             'chrom'   : info['chrom'],
             'position': info['position']}
            for p, info in probe_dmb_map.items()
            if p in tcga_meth.index
        ])
        print(f'Probes to map: {len(probe_pos_df):,}')

        # For each chromosome: merge_asof to find nearest CT window
        # merge_asof requires sorted keys — sort both tables by position/start
        ct_sorted = ct_df[['chrom','start','ct_z']]\
                        .sort_values(['chrom','start'])
        probe_sorted = probe_pos_df.sort_values(['chrom','position'])

        matched_parts = []
        for chrom in probe_sorted['chrom'].unique():
            p_chr = probe_sorted[probe_sorted['chrom']==chrom]
            c_chr = ct_sorted[ct_sorted['chrom']==chrom]
            if len(c_chr) == 0: continue
            merged = pd.merge_asof(
                p_chr.rename(columns={'position':'pos'}),
                c_chr.rename(columns={'start':'pos'})[['pos','ct_z']],
                on='pos', direction='nearest'
            )
            matched_parts.append(merged)

        if not matched_parts:
            print('No CT-probe matches found.')
        else:
            matched = pd.concat(matched_parts, ignore_index=True)
            matched = matched.dropna(subset=['ct_z'])
            print(f'Probes matched to CT windows: {len(matched):,}')

            # Align with TCGA variance
            probe_var_s = probe_var.reindex(matched['probe_id'].values)
            valid = probe_var_s.notna()
            ct_z_aligned  = matched.loc[valid.values, 'ct_z'].values
            var_aligned   = probe_var_s[valid].values
            probe_ids_out = matched.loc[valid.values, 'probe_id'].values

            r_b, p_b = spearmanr(ct_z_aligned, var_aligned)
            print(f'\nSpearman r (CT-z vs TCGA var): {r_b:.4f}')
            print(f'p-value                       : {p_b:.3e}')
            print(f'Probes in correlation         : {len(var_aligned):,}')

            if abs(r_b) > 0.3 and p_b < 0.01:
                print('RESULT: ✔ V6 CT instability significantly '
                      'predicts TCGA variance')
            elif p_b < 0.05:
                print('RESULT: ~ Weak but significant correlation')
            else:
                print('RESULT: ✗ No significant CT-variance correlation')

            layerB = pd.DataFrame({
                'probe_id'     : probe_ids_out,
                'v6_ct_score_z': ct_z_aligned,
                'tcga_variance' : var_aligned,
            })
            layerB.to_csv(f'{OUT_DIR}/layerB_ct_replication.csv',
                          index=False)
            print('Saved: layerB_ct_replication.csv')


=== Layer B: V6 CT Instability Score vs TCGA Variance ===
CT windows loaded: 2,145,652 across 24 chromosomes
CT column   : ct_score_raw
CT-z range  : [-1.68, 5.66]
Probes to map: 105,451
Probes matched to CT windows: 105,451

Spearman r (CT-z vs TCGA var): 0.0126
p-value                       : 4.111e-05
Probes in correlation         : 105,451
RESULT: ~ Weak but significant correlation
Saved: layerB_ct_replication.csv


## Cell 6 — Layer C: Δβ Direction Concordance (Tumour vs Normal)
TCGA barcode suffix: `-01` to `-06` = tumour; `-10` to `-12` = solid normal tissue.
Expected: V6 Δβ(IMR90−H1) anti-correlates with TCGA Δβ(tumour−normal).

In [8]:
print('=== Layer C: V6 Δβ Direction vs TCGA Tumour/Normal ===')

tumour_cols = [c for c in tcga_meth.columns
               if len(c) >= 15 and c[13:15] in
               ['01','02','03','04','05','06']]
normal_cols = [c for c in tcga_meth.columns
               if len(c) >= 15 and c[13:15] in ['10','11','12']]

print(f'Tumour samples : {len(tumour_cols):,}')
print(f'Normal samples : {len(normal_cols):,}')
print(f'Total samples  : {len(tcga_meth.columns):,}')

concordance_pct = np.nan
r_c, p_c = np.nan, np.nan
layerC = pd.DataFrame()

if len(tumour_cols) > 10 and len(normal_cols) > 5:
    mean_T = tcga_meth[tumour_cols].mean(axis=1)
    mean_N = tcga_meth[normal_cols].mean(axis=1)
    tcga_d = mean_T - mean_N  # tumour - normal

    v6_d_map = {p: info['v6_delta']
                for p, info in probe_dmb_map.items()
                if p in tcga_d.index}
    v6_d   = pd.Series(v6_d_map)
    tcga_d = tcga_d.reindex(v6_d.index).dropna()
    v6_d   = v6_d.reindex(tcga_d.index)

    near_zero_n = (tcga_d.abs() < 0.01).sum()
    print(f'Probes with both Δβ values : {len(tcga_d):,}')
    print(f'Near-zero TCGA Δβ (<0.01) : {near_zero_n} ({near_zero_n/len(tcga_d)*100:.0f}%)')

    # Anti-correlation: V6 Δβ>0 (hyper-IMR90) → tumour hypomethylated vs normal
    concordant = ((v6_d>0)&(tcga_d<0)) | ((v6_d<0)&(tcga_d>0))
    concordance_pct = concordant.mean() * 100
    r_c, p_c = spearmanr(-v6_d, tcga_d)

    print(f'Direction concordance      : {concordance_pct:.2f}%')
    print(f'Spearman r(-V6Δβ, TCGAΔβ) : {r_c:.4f}  p={p_c:.3e}')

    if concordance_pct > 60 and p_c < 0.01:
        print('RESULT: ✔ STRONG concordance')
    elif concordance_pct > 55:
        print('RESULT: ~ Moderate concordance')
    else:
        print('RESULT: ✗ Poor concordance')

    layerC = pd.DataFrame({
        'probe_id'  : tcga_d.index,
        'v6_delta'  : v6_d.values,
        'tcga_delta': tcga_d.values,
        'concordant': concordant.values,
    })
    layerC.to_csv(f'{OUT_DIR}/layerC_direction_concordance.csv', index=False)
    print('Saved: layerC_direction_concordance.csv')
else:
    print(f'Insufficient normal samples ({len(normal_cols)}) for Layer C.')
    print('All TCGA samples may be tumour-type in tcga_dmb_probes_full.tsv.')
    print('Layer C skipped. Layers A and B are still fully valid.')


=== Layer C: V6 Δβ Direction vs TCGA Tumour/Normal ===
Tumour samples : 8,695
Normal samples : 671
Total samples  : 9,854
Probes with both Δβ values : 105,451
Near-zero TCGA Δβ (<0.01) : 12569 (12%)
Direction concordance      : 40.83%
Spearman r(-V6Δβ, TCGAΔβ) : -0.2479  p=0.000e+00
RESULT: ✗ Poor concordance
Saved: layerC_direction_concordance.csv


## Cell 7 — Per-chromosome validation summary

In [9]:
print('=== Per-chromosome Validation ===')

chr_results = []
for chrom in CHROMS:
    chr_dmb = [p for p, info in probe_dmb_map.items()
               if info['chrom']==chrom and p in tcga_meth.index]
    if len(chr_dmb) < 5: continue

    v_dmb = probe_var.reindex(chr_dmb).dropna()
    v_bg  = probe_var.drop(index=chr_dmb, errors='ignore').dropna()
    fold  = v_dmb.median() / max(v_bg.median(), 1e-9)

    conc = np.nan
    if len(layerC) > 0:
        sub = layerC[layerC['probe_id'].isin(chr_dmb)]
        if len(sub) > 0:
            conc = sub['concordant'].mean() * 100

    _, pv = mannwhitneyu(v_dmb, v_bg, alternative='greater') \
            if len(v_dmb)>5 and len(v_bg)>5 \
            else (np.nan, np.nan)

    chr_results.append({
        'chrom'          : chrom,
        'n_dmb_probes'   : len(chr_dmb),
        'median_var'     : round(float(v_dmb.median()), 6),
        'enrichment_fold': round(fold, 3),
        'mw_pvalue'      : float(pv) if not np.isnan(pv) else np.nan,
        'concordance_pct': round(conc,2) if not np.isnan(conc) else np.nan,
    })

chr_df = pd.DataFrame(chr_results)
chr_df.to_csv(f'{OUT_DIR}/per_chromosome_validation.csv', index=False)

print(chr_df.to_string(index=False))
print()
n_sig = (chr_df['enrichment_fold'] >= 1.3).sum()
print(f'Chroms with fold ≥ 1.3 : {n_sig}/{len(chr_df)}')
best = chr_df.loc[chr_df['enrichment_fold'].idxmax()]
print(f'Best chrom             : {best["chrom"]} '
      f'(fold={best["enrichment_fold"]:.2f}x, '
      f'n={best["n_dmb_probes"]:,})')
print('Saved: per_chromosome_validation.csv')


=== Per-chromosome Validation ===
chrom  n_dmb_probes  median_var  enrichment_fold    mw_pvalue  concordance_pct
 chr1         10388    0.030466            1.004 1.063690e-01            40.16
 chr2          6768    0.030592            1.008 4.517641e-02            38.52
 chr3          4685    0.029776            0.980 8.760246e-01            44.18
 chr4          3494    0.031082            1.025 1.369254e-02            37.75
 chr5          5602    0.033988            1.128 2.301679e-39            46.91
 chr6          8161    0.028703            0.942 1.000000e+00            40.93
 chr7          6631    0.034527            1.149 8.911251e-62            34.10
 chr8          4516    0.034346            1.139 1.543493e-42            30.80
 chr9          2079    0.025086            0.823 1.000000e+00            47.09
chr10          5485    0.033615            1.115 1.332013e-31            37.37
chr11          5954    0.030123            0.992 4.682032e-01            35.56
chr12          521

## Cell 8 — 4-panel validation figure

In [10]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.patch.set_facecolor('#F8F9FA')
fig.suptitle(
    'Meth3D-Net V6 — TCGA PanCancer Validation\n'
    f'H1 ESC vs IMR90 DMBs | {len(tcga_meth):,} probes × '
    f'{len(tcga_meth.columns):,} TCGA samples | 33 cancer types',
    fontsize=13, fontweight='bold', y=1.01
)

# Panel A — Variance enrichment violin (High vs Low |Δβ|)
ax = axes[0,0]
vp = ax.violinplot(
    [list(var_low.sample(min(5000,len(var_low)),random_state=42)),
     list(var_high)],
    positions=[0,1], showmedians=True, showextrema=False
)
vp['bodies'][0].set_facecolor('#74B9FF'); vp['bodies'][0].set_alpha(0.7)
vp['bodies'][1].set_facecolor('#E17055'); vp['bodies'][1].set_alpha(0.7)
vp['cmedians'].set_color('black'); vp['cmedians'].set_linewidth(2)
ax.set_xticks([0,1])
ax.set_xticklabels(['Low V6 |Δβ|\n(<0.30)',
                    'High V6 |Δβ|\n(≥0.30)'])
ax.set_ylabel(f'TCGA methylation variance ({len(tcga_meth.columns):,} samples)')
ax.set_title(
    f'A  Variance Enrichment\n'
    f'Fold={fold_enrich:.2f}x  p={pval:.1e}',
    fontweight='bold', loc='left'
)
ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.spines[['top','right']].set_visible(False)

# Panel B — CT score vs TCGA variance
ax2 = axes[0,1]
if len(layerB) > 0 and 'v6_ct_score_z' in layerB.columns:
    sb = layerB.sample(min(3000,len(layerB)), random_state=42)
    ax2.scatter(sb['v6_ct_score_z'], sb['tcga_variance'],
                alpha=0.25, s=6, color='#6C5CE7')
    ax2.set_xlabel('V6 CT Instability Score (z)')
    ax2.set_ylabel('TCGA Methylation Variance')
    lbl = f'B  CT Score vs TCGA Variance\n'\
          f'Spearman r={r_b:.3f}  p={p_b:.1e}'
else:
    lbl = 'B  CT Score vs TCGA Variance\n(CT files not found)'
    ax2.text(0.5,0.5,'CT data not available',ha='center',
             va='center',transform=ax2.transAxes)
ax2.set_title(lbl, fontweight='bold', loc='left')
ax2.grid(alpha=0.2); ax2.spines[['top','right']].set_visible(False)

# Panel C — per-chromosome bar chart
ax3 = axes[1,0]
colors_c = ['#00B894' if f>=1.5 else '#FDCB6E' if f>=1.0 else '#FF7675'
            for f in chr_df['enrichment_fold']]
ax3.bar(range(len(chr_df)), chr_df['enrichment_fold'],
        color=colors_c, alpha=0.85, edgecolor='white')
ax3.axhline(1.0, color='black', ls='--', lw=1, alpha=0.5, label='No enrichment')
ax3.axhline(1.5, color='#00B894', ls=':', lw=1.5, alpha=0.8, label='1.5x')
ax3.set_xticks(range(len(chr_df)))
ax3.set_xticklabels([c.replace('chr','') for c in chr_df['chrom']],
                     fontsize=8, rotation=45)
ax3.set_ylabel('Variance enrichment fold\n(V6 DMB vs background probes)')
ax3.set_title('C  Per-chromosome Enrichment', fontweight='bold', loc='left')
ax3.legend(fontsize=8, frameon=False)
ax3.spines[['top','right']].set_visible(False)

# Panel D — direction concordance
ax4 = axes[1,1]
if len(layerC) > 0:
    sc = layerC.sample(min(3000,len(layerC)), random_state=42)
    cm = sc['concordant'].astype(bool)
    ax4.scatter(sc.loc[cm,'v6_delta'], sc.loc[cm,'tcga_delta'],
                alpha=0.25, s=6, color='#00B894',
                label=f'Concordant ({concordance_pct:.1f}%)')
    ax4.scatter(sc.loc[~cm,'v6_delta'], sc.loc[~cm,'tcga_delta'],
                alpha=0.25, s=6, color='#FF7675', label='Discordant')
    ax4.axhline(0, color='black', lw=0.8, alpha=0.4)
    ax4.axvline(0, color='black', lw=0.8, alpha=0.4)
    ax4.set_xlabel('V6 Δβ (IMR90 − H1 ESC)')
    ax4.set_ylabel('TCGA Δβ (tumour − normal)')
    ax4.set_title(
        f'D  Direction Concordance (Tumour vs Normal)\n'
        f'r={r_c:.3f}  concordance={concordance_pct:.1f}%',
        fontweight='bold', loc='left'
    )
    ax4.legend(fontsize=8, frameon=False)
else:
    ax4.text(0.5, 0.5,
             'Tumour vs Normal comparison\nnot available\n'
             f'(normal samples in dataset: {len(normal_cols)})',
             ha='center', va='center', transform=ax4.transAxes,
             fontsize=10, color='#636e72')
    ax4.set_title('D  Direction Concordance', fontweight='bold', loc='left')
ax4.spines[['top','right']].set_visible(False)

plt.tight_layout()
for ext in ['pdf','png']:
    fp = f'{OUT_DIR}/Fig_V6_TCGA_Validation.{ext}'
    plt.savefig(fp, dpi=300 if ext=='pdf' else 150, bbox_inches='tight')
plt.close()
print('Saved: Fig_V6_TCGA_Validation.pdf + .png')


Saved: Fig_V6_TCGA_Validation.pdf + .png


## Cell 9 — Validation summary

In [11]:
print('='*65)
print('  METH3D-NET V6 — TCGA PANCANCAN VALIDATION SUMMARY')
print('='*65)
print(f'  Dataset            : TCGA PanCancer 33 cancer types')
print(f'  V6 DMBs (sig.)     : {len(sig_dmb):,}')
print(f'  Probe-DMB overlaps : {len(DMB_PROBE_SET):,}')
print(f'  TCGA probes loaded : {len(tcga_meth):,}')
print(f'  TCGA samples       : {len(tcga_meth.columns):,}')
print()
print('  LAYER A — Variance Enrichment')
print(f'    Fold enrichment  : {fold_enrich:.3f}x')
print(f'    Mann-Whitney p   : {pval:.3e}')
print(f'    Result           : {"SIGNIFICANT" if pval<0.01 else "NOT significant"}')
print()
print('  LAYER B — CT Instability Replication')
if not np.isnan(r_b):
    print(f'    Spearman r (CT-z): {r_b:.4f}')
    print(f'    p-value          : {p_b:.3e}')
    print(f'    Result           : {"SIGNIFICANT" if p_b<0.05 else "NOT significant"}')
else:
    print('    CT files not found — Layer B skipped')
print()
print('  LAYER C — Direction Concordance (Tumour vs Normal)')
if not np.isnan(concordance_pct):
    print(f'    Tumour samples   : {len(tumour_cols):,}')
    print(f'    Normal samples   : {len(normal_cols):,}')
    print(f'    Concordance      : {concordance_pct:.2f}%')
    print(f'    Spearman r       : {r_c:.4f}  p={p_c:.3e}')
else:
    print(f'    Skipped — only {len(normal_cols)} normal samples found')
print()
print('  PER-CHROMOSOME:')
top5 = chr_df.nlargest(5,'enrichment_fold')[['chrom','n_dmb_probes','enrichment_fold']]
for _, row in top5.iterrows():
    print(f'    {row["chrom"]:<6} : {row["n_dmb_probes"]:>6,} probes  '
          f'fold={row["enrichment_fold"]:.2f}x')
print()
print('  Output files:')
for fn in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(f'{OUT_DIR}/{fn}')
    unit = 'MB' if sz>1e6 else 'KB'
    szv = sz/1e6 if sz>1e6 else sz/1024
    print(f'    {fn:<50} {szv:>6.1f} {unit}')
print('='*65)


# Copy all outputs to Google Drive for persistent storage
import shutil
print('\nCopying outputs to Drive...')
for fn in sorted(os.listdir(OUT_DIR)):
    src_path = os.path.join(OUT_DIR, fn)
    sz = os.path.getsize(src_path)/1024
    print(f'  {fn:<50} {sz:>6.1f} KB')
print(f'\nAll outputs at: {OUT_DIR}')


  METH3D-NET V6 — TCGA PANCANCAN VALIDATION SUMMARY
  Dataset            : TCGA PanCancer 33 cancer types
  V6 DMBs (sig.)     : 77,066
  Probe-DMB overlaps : 134,178
  TCGA probes loaded : 105,451
  TCGA samples       : 9,854

  LAYER A — Variance Enrichment
    Fold enrichment  : 1.359x
    Mann-Whitney p   : 0.000e+00
    Result           : SIGNIFICANT

  LAYER B — CT Instability Replication
    Spearman r (CT-z): 0.0126
    p-value          : 4.111e-05
    Result           : SIGNIFICANT

  LAYER C — Direction Concordance (Tumour vs Normal)
    Tumour samples   : 8,695
    Normal samples   : 671
    Concordance      : 40.83%
    Spearman r       : -0.2479  p=0.000e+00

  PER-CHROMOSOME:
    chrX   :  1,666 probes  fold=1.20x
    chr7   :  6,631 probes  fold=1.15x
    chr8   :  4,516 probes  fold=1.14x
    chr5   :  5,602 probes  fold=1.13x
    chr10  :  5,485 probes  fold=1.11x

  Output files:
    Fig_V6_TCGA_Validation.pdf                          140.8 KB
    Fig_V6_TCGA_Validati